# Week 08 - Day 06 | Testing with pytest

**Topics:** Test discovery · plain `assert` · `pytest.raises` · fixtures · parametrize · coverage basics

---

## pytest Cheatsheet

| Concept | Syntax |
|---|---|
| Discovery | files `test_*.py`, functions `test_*`, classes `Test*` |
| Assertion | plain `assert a == b` — pytest shows both sides on failure |
| Expected exception | `with pytest.raises(ValueError): ...` |
| Fixture | `@pytest.fixture` → injected into tests by parameter name |
| Parametrize | `@pytest.mark.parametrize("x, y", [(1, 2), (3, 4)])` |
| Run | `pytest file.py -v` |
| Coverage | `pytest --cov=module` (needs `pytest-cov`) |

> Install with `pip install pytest` — already in this repo's `requirements.txt`.

In [ ]:
# 1. Code under test + plain assert tests

import pytest

def add_item(cart: dict[str, int], name: str, qty: int = 1) -> None:
    """Add qty of an item to the cart (must be positive)."""
    if qty <= 0:
        raise ValueError("qty must be positive")
    cart[name] = cart.get(name, 0) + qty

def cart_total(cart: dict[str, int], prices: dict[str, float]) -> float:
    return sum(prices[name] * qty for name, qty in cart.items())

def test_add_item_new():
    cart: dict[str, int] = {}
    add_item(cart, "apple", 3)
    assert cart == {"apple": 3}

def test_cart_total():
    assert cart_total({"apple": 2, "bread": 1}, {"apple": 0.5, "bread": 2.0}) == 3.0

# pytest would collect these automatically; here we run them manually:
test_add_item_new()
test_cart_total()
print("plain assert tests passed ✅")

In [ ]:
# 2. Testing exceptions — pytest.raises

def test_add_item_rejects_zero_qty():
    with pytest.raises(ValueError):
        add_item({}, "apple", 0)

def test_add_item_error_message():
    with pytest.raises(ValueError, match="positive"):  # message must match too
        add_item({}, "apple", -5)

test_add_item_rejects_zero_qty()
test_add_item_error_message()
print("exception tests passed ✅")

In [ ]:
# 3. Fixtures — reusable setup injected by parameter name

@pytest.fixture
def stocked_cart() -> dict[str, int]:
    """A cart that already contains a few items."""
    return {"apple": 2, "bread": 1, "milk": 4}

def test_add_to_stocked(stocked_cart):   # pytest injects the fixture here
    add_item(stocked_cart, "milk", 1)
    assert stocked_cart["milk"] == 5

# Fixtures are injected by the pytest RUNNER — in a notebook we simulate it:
test_add_to_stocked({"apple": 2, "bread": 1, "milk": 4})
print("fixture-style test passed ✅ (run `pytest lesson.py -v` for real injection)")

In [ ]:
# 4. Parametrize — one test, many cases

@pytest.mark.parametrize("qty, expected", [
    (1, 1),
    (3, 3),
    (10, 10),
])
def test_add_item_quantities(qty, expected):
    cart: dict[str, int] = {}
    add_item(cart, "apple", qty)
    assert cart["apple"] == expected

# pytest expands this into 3 separate test results; manual simulation:
for qty, expected in [(1, 1), (3, 3), (10, 10)]:
    test_add_item_quantities(qty, expected)
print("parametrized cases passed ✅")

## Running pytest

```bash
pytest lesson.py -v      # verbose — one line per test
pytest -k "total"        # only tests matching a name
pytest -x                # stop at first failure
pytest --lf              # re-run only last failures
pytest --cov=lesson      # coverage report (pip install pytest-cov)
```

Coverage shows **which lines your tests executed** — 100% coverage ≠ bug-free,
it only means every line ran at least once.

## Summary

| Concept | Key Point |
|---|---|
| Discovery | `test_*.py` files / `test_*` functions found automatically |
| `assert` | Plain assert — pytest reports both sides of a failed comparison |
| `pytest.raises` | Passes only if the expected exception is raised |
| Fixtures | Setup functions injected into tests by parameter name, fresh per test |
| Parametrize | Same test body, many cases, reported individually |
| Coverage | `pytest --cov` — measures which lines tests executed |

> **Next:** Day 07 — Weekly Review (10 MCQ + 5 code challenges + weekly project)